# Stage 6 — Frozen Held-Out Cross-Domain Evaluation

**Evaluation only — no training.**

This notebook evaluates the six completed YOLO11m Stage 5 final-epoch checkpoints (`last.pt`) on both frozen held-out test domains, producing exactly 12 evaluation cells.

Frozen integrity rules:
- no training or fine-tuning;
- no checkpoint selection after test inspection;
- no threshold tuning on held-out test;
- RF-DETR interrupted runs are excluded;
- Stage 6 code is taken from the `stage6-heldout-eval` branch.

In [2]:
# Cell 1 — Mount Drive and GPU preflight
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
RUN_ROOT = DRIVE / 'Stage5_Baselines'
OUT = DRIVE / 'Stage6_Heldout_Evaluation'
OUT.mkdir(parents=True, exist_ok=True)

required_runs = [
    'Y_YOLO_s17','Y_YOLO_s42','Y_YOLO_s2026',
    'C_YOLO_s17','C_YOLO_s42','C_YOLO_s2026',
]
missing = [r for r in required_runs if not (RUN_ROOT/r).is_dir()]
if missing:
    raise FileNotFoundError('Missing completed Stage 5 run folder(s): ' + ', '.join(missing))

if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU attached. Select a GPU runtime before executing Stage 6.')
subprocess.run(['nvidia-smi'], check=True)
print('STAGE6_ENVIRONMENT_PREFLIGHT: PASS')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RuntimeError: No NVIDIA GPU attached. Select a GPU runtime before executing Stage 6.

In [ ]:
# Cell 2 — Reassemble and verify the frozen h9-v2 data bundle
import hashlib, zipfile, shutil
from pathlib import Path

PARTS = [
    DRIVE / 'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_00',
    DRIVE / 'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_01',
    DRIVE / 'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_02',
]
missing = [str(p) for p in PARTS if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing frozen-data bundle part(s): ' + ', '.join(missing))

BUNDLE = Path('/content/Stage5_Frozen_h9_v2_Data_Bundle.zip')
EXPECTED_SHA = 'dd8ce20f2650848133ce4323b00139684b0d4ac8b7db618e8df469244221cee9'
h = hashlib.sha256()
with BUNDLE.open('wb') as out:
    for part in PARTS:
        with part.open('rb') as f:
            for chunk in iter(lambda: f.read(8*1024*1024), b''):
                out.write(chunk); h.update(chunk)
assert h.hexdigest() == EXPECTED_SHA, (h.hexdigest(), EXPECTED_SHA)

DATA_ROOT = Path('/content/stage6_frozen_data')
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError('Corrupt ZIP member: ' + bad)
    zf.extractall(DATA_ROOT)

Y_DATA = DATA_ROOT / 'h9_v2/Y-PPE-h9-v2/data.yaml'
C_DATA = DATA_ROOT / 'h9_v2/Construction-PPE-h9-v2/data.yaml'
assert Y_DATA.is_file() and C_DATA.is_file()
print('FROZEN_DATA_REASSEMBLY: PASS')

In [ ]:
# Cell 3 — Clone the frozen Stage 6 evaluation branch
import subprocess, shutil
from pathlib import Path

REPO = Path('/content/Y-PPE-Detection-RFDETR')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git','clone','--branch','stage6-heldout-eval','--single-branch',
    'https://github.com/EZZALDEEN049/Y-PPE-Detection-RFDETR.git', str(REPO)
], check=True)
STAGE6_SHA = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
(OUT/'stage6_code_sha.txt').write_text(STAGE6_SHA+'\n', encoding='utf-8')
print('Stage 6 code SHA:', STAGE6_SHA)

In [ ]:
# Cell 4 — Install the exact YOLO evaluation application layer
import subprocess, sys
subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'ultralytics==8.4.156','PyYAML==6.0.3','pandas'
], check=True)

import ultralytics, torch
print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
assert ultralytics.__version__ == '8.4.156'
assert torch.cuda.is_available()
print('PINNED_EVALUATION_ENV: PASS')

In [ ]:
# Cell 5 — Stage 6 preflight ONLY (does not run held-out inference)
import subprocess, sys
EVAL = REPO / 'cross_domain_reliability/stage6_heldout_evaluation/evaluate_yolo_cross_domain.py'
cmd = [
    sys.executable, str(EVAL),
    '--stage5-root', str(RUN_ROOT),
    '--y-data', str(Y_DATA),
    '--c-data', str(C_DATA),
    '--out', str(OUT),
    '--device', '0',
    '--batch', '8',
    '--imgsz', '640',
    '--ap-conf', '0.001',
    '--operating-conf', '0.25',
    '--nms-iou', '0.70',
    '--match-iou', '0.50',
    '--preflight-only',
]
subprocess.run(cmd, check=True)
print('NO HELD-OUT INFERENCE HAS RUN IN THIS CELL.')

## Frozen execution gate

If Cell 5 prints `STAGE6_PREFLIGHT_PASS`, all six completed YOLO11m final-epoch checkpoints, both frozen dataset fingerprints, and the exact held-out test counts have passed the integrity gate.

The next cell begins the **first Stage 6 held-out evaluation**. Its settings are already frozen and must not be modified after results are seen.

In [ ]:
# Cell 6 — Execute all 12 frozen held-out evaluation cells
import subprocess, sys, time
start = time.time()
cmd = [
    sys.executable, str(EVAL),
    '--stage5-root', str(RUN_ROOT),
    '--y-data', str(Y_DATA),
    '--c-data', str(C_DATA),
    '--out', str(OUT),
    '--device', '0',
    '--batch', '8',
    '--imgsz', '640',
    '--ap-conf', '0.001',
    '--operating-conf', '0.25',
    '--nms-iou', '0.70',
    '--match-iou', '0.50',
]
subprocess.run(cmd, check=True)
print(f'STAGE6_12CELL_EVALUATION_COMPLETE in {(time.time()-start)/60:.1f} min')

In [ ]:
# Cell 7 — Compute seed statistics and paired domain-generalization gaps
import subprocess, sys
SUM = REPO / 'cross_domain_reliability/stage6_heldout_evaluation/summarize_stage6.py'
subprocess.run([
    sys.executable, str(SUM),
    '--input', str(OUT/'stage6_12cell_summary.csv'),
    '--out-dir', str(OUT),
], check=True)
print('STAGE6_SUMMARIZATION: PASS')

In [ ]:
# Cell 8 — Display the primary result tables
import pandas as pd
from IPython.display import display

raw = pd.read_csv(OUT/'stage6_12cell_summary.csv')
gaps = pd.read_csv(OUT/'stage6_seed_domain_gaps.csv')
gap_summary = pd.read_csv(OUT/'stage6_domain_gap_summary.csv')

print('12-cell frozen held-out results')
display(raw)
print('\nPaired seed-level domain gaps')
display(gaps)
print('\nMean ± SD domain gaps by training direction')
display(gap_summary)

In [ ]:
# Cell 9 — Write Stage 6 completion record
import json
from datetime import datetime, timezone

status = {
    'stage': 6,
    'status': 'STAGE6_12CELL_HELDOUT_EVALUATION_COMPLETE',
    'evaluation_only': True,
    'training_performed': False,
    'eligible_models': [
        'Y_YOLO_s17','Y_YOLO_s42','Y_YOLO_s2026',
        'C_YOLO_s17','C_YOLO_s42','C_YOLO_s2026',
    ],
    'checkpoint_policy': 'final_epoch_last.pt',
    'n_evaluation_cells': 12,
    'stage6_code_sha': STAGE6_SHA,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
(OUT/'stage6_status.json').write_text(json.dumps(status, indent=2), encoding='utf-8')
print(json.dumps(status, indent=2))